## SELECCIÓN DEL MEJOR MODELO (Manual)

### By: Cristian David Ceballos Velez  
### Date: 2026-08-26  
### Description: Selección del mejor modelo de ML para predicción de enfermedad cardiaca, comparado contra baseline heurístico.


## 1. Importar librerías

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import friedmanchisquare, randint, loguniform, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Cargar y preparar datos

Se utiliza el dataset base en `data/01_raw/corazon.csv`. Como el archivo contiene algunos registros inválidos en variables numéricas y en el target, se realiza limpieza mínima para conservar solo observaciones válidas.

In [ ]:
ROOT_DIR = Path.cwd().resolve().parents[1]
DATA_PATH = ROOT_DIR / "data" / "01_raw" / "corazon.csv"

df = pd.read_csv(DATA_PATH)

TARGET = "disease"
num_cols = ["age", "rest_bp", "chol", "fbs", "max_hr", "exang", "old_peak", "slope", "ca"]
cat_cols = ["sex", "chest_pain", "rest_ecg", "thal"]

for col in num_cols + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# mantener únicamente etiqueta binaria válida
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)

print(df.shape)
print(df[TARGET].value_counts())
df.head()

## 3. Split Train/Test

Se separa el conjunto de prueba antes de entrenar para evitar fuga de información.

In [ ]:
X = df[num_cols + cat_cols].copy()
y = df[TARGET].copy()

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape, y_test.shape)

## 4. Pipeline de preprocesamiento (reutilizable)

In [ ]:
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categoric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categoric_pipe, cat_cols),
    ]
)

preprocessor

## 5. Selección de múltiples modelos (mínimo 4)

Métrica principal: **Recall** (reducir falsos negativos de enfermedad).  
También se monitorean Accuracy, Precision y F1.

In [ ]:
candidate_models = {
    "LogisticRegression": LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVC": SVC(random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "ExtraTrees": ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "recall": "recall",
    "f1": "f1",
    "accuracy": "accuracy",
    "precision": "precision",
}

cv_rows = []
for name, model in candidate_models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_validate(pipe, x_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append(
        {
            "model": name,
            **{f"{m}_mean": scores[f"test_{m}"].mean() for m in scoring},
            **{f"{m}_std": scores[f"test_{m}"].std() for m in scoring},
        }
    )

cv_results = pd.DataFrame(cv_rows).sort_values("recall_mean", ascending=False).reset_index(drop=True)
cv_results

## 6. Eliminar modelos por debajo del promedio

Se descartan los modelos con Recall promedio por debajo de la media grupal.

In [ ]:
recall_threshold = cv_results["recall_mean"].mean()
survivors = cv_results[cv_results["recall_mean"] >= recall_threshold].copy()

print("Recall promedio grupal:", round(recall_threshold, 4))
print("Modelos seleccionados:", survivors["model"].tolist())
survivors

## 7. Optimización de hiperparámetros (Top 3)

Se optimizan los 3 mejores modelos por Recall usando `RandomizedSearchCV` + validación cruzada.

In [ ]:
param_distributions = {
    "LogisticRegression": {
        "model__C": loguniform(1e-3, 1e2),
        "model__penalty": ["l1", "l2"],
    },
    "RandomForest": {
        "model__n_estimators": randint(100, 500),
        "model__max_depth": randint(3, 30),
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
    },
    "GradientBoosting": {
        "model__n_estimators": randint(50, 300),
        "model__learning_rate": loguniform(1e-3, 5e-1),
        "model__max_depth": randint(2, 6),
        "model__subsample": np.linspace(0.6, 1.0, 30),
    },
    "SVC": {
        "model__C": loguniform(1e-2, 1e2),
        "model__gamma": ["scale", "auto"],
        "model__kernel": ["rbf", "linear"],
    },
    "KNN": {
        "model__n_neighbors": randint(3, 35),
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
    "ExtraTrees": {
        "model__n_estimators": randint(100, 500),
        "model__max_depth": randint(3, 30),
        "model__min_samples_split": randint(2, 20),
        "model__min_samples_leaf": randint(1, 10),
    },
}

top3_names = survivors.head(3)["model"].tolist()
search_results = []
best_estimators = {}

for name in top3_names:
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", candidate_models[name])])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions[name],
        n_iter=20,
        scoring="recall",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    search.fit(x_train, y_train)

    best_estimators[name] = search.best_estimator_

    y_pred_test = search.best_estimator_.predict(x_test)
    search_results.append(
        {
            "model": name,
            "best_cv_recall": search.best_score_,
            "test_recall": recall_score(y_test, y_pred_test),
            "test_f1": f1_score(y_test, y_pred_test),
            "test_accuracy": accuracy_score(y_test, y_pred_test),
            "test_precision": precision_score(y_test, y_pred_test),
            "best_params": search.best_params_,
        }
    )

tuned_df = pd.DataFrame(search_results).sort_values("best_cv_recall", ascending=False).reset_index(drop=True)
tuned_df

## 8. Comparación estadística de los modelos optimizados

Se compara Recall por fold (5 folds) para los 3 modelos optimizados usando Friedman y comparaciones pareadas Wilcoxon.

In [ ]:
fold_scores = {}
for name in top3_names:
    scores = cross_validate(best_estimators[name], x_train, y_train, cv=cv, scoring="recall", n_jobs=-1)
    fold_scores[name] = scores["test_score"]

stats_df = pd.DataFrame(fold_scores)
print(stats_df)

friedman_stat, friedman_p = friedmanchisquare(*[stats_df[col] for col in stats_df.columns])
print(f"Friedman statistic={friedman_stat:.4f}, p-value={friedman_p:.6f}")

pairwise = []
cols = list(stats_df.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        w_stat, p_val = wilcoxon(stats_df[cols[i]], stats_df[cols[j]], zero_method="zsplit")
        pairwise.append({"model_a": cols[i], "model_b": cols[j], "wilcoxon_p": p_val})

pd.DataFrame(pairwise).sort_values("wilcoxon_p")

## 9. Seleccionar mejor modelo y evaluar en test

Baseline previo (notebook 5): **Recall = 0.97** para la clase positiva (`disease=1`).

In [ ]:
best_row = tuned_df.iloc[0]
best_model_name = best_row["model"]
best_pipeline = best_estimators[best_model_name]

best_pipeline.fit(x_train, y_train)
y_pred_test = best_pipeline.predict(x_test)

baseline_recall = 0.97
final_metrics = {
    "best_model": best_model_name,
    "test_recall": recall_score(y_test, y_pred_test),
    "test_f1": f1_score(y_test, y_pred_test),
    "test_accuracy": accuracy_score(y_test, y_pred_test),
    "test_precision": precision_score(y_test, y_pred_test),
    "baseline_recall": baseline_recall,
    "improvement_vs_baseline": recall_score(y_test, y_pred_test) - baseline_recall,
}

pd.Series(final_metrics)

In [ ]:
print(classification_report(y_test, y_pred_test))

## 10. Overfitting / Underfitting

Se evalúa brecha entre recall de entrenamiento y recall de test para el modelo final.

In [ ]:
y_pred_train = best_pipeline.predict(x_train)
train_recall = recall_score(y_train, y_pred_train)
test_recall = recall_score(y_test, y_pred_test)
gap = train_recall - test_recall

print(f"Train recall: {train_recall:.4f}")
print(f"Test recall : {test_recall:.4f}")
print(f"Gap         : {gap:.4f}")

if gap > 0.05:
    print("Posible sobreajuste (train mucho mayor que test).")
elif test_recall < 0.80:
    print("Posible underfitting (desempeño bajo en test).")
else:
    print("No se observan señales severas de overfitting/underfitting con la métrica principal.")

## 11. Learning Curve y escalabilidad

In [ ]:
train_sizes, train_scores, val_scores, fit_times, score_times = learning_curve(
    estimator=best_pipeline,
    X=x_train,
    y=y_train,
    cv=cv,
    train_sizes=np.linspace(0.1, 1.0, 5),
    scoring="recall",
    n_jobs=-1,
    return_times=True,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)
fit_mean = fit_times.mean(axis=1)
fit_std = fit_times.std(axis=1)
score_mean = score_times.mean(axis=1)
score_std = score_times.std(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_sizes, train_mean, "o-", label="Train Recall")
axes[0].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
axes[0].plot(train_sizes, val_mean, "o-", label="CV Recall")
axes[0].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
axes[0].set_title("Learning Curve")
axes[0].set_xlabel("Tamaño de entrenamiento")
axes[0].set_ylabel("Recall")
axes[0].legend()

axes[1].plot(train_sizes, fit_mean, "o-", label="Fit time")
axes[1].fill_between(train_sizes, fit_mean - fit_std, fit_mean + fit_std, alpha=0.2)
axes[1].plot(train_sizes, score_mean, "o-", label="Score time")
axes[1].fill_between(train_sizes, score_mean - score_std, score_mean + score_std, alpha=0.2)
axes[1].set_title("Escalabilidad (tiempos)")
axes[1].set_xlabel("Tamaño de entrenamiento")
axes[1].set_ylabel("Tiempo (s)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Guardar pipeline final

Se almacena el pipeline de preprocesamiento + modelo para inferencia.

In [ ]:
MODEL_PATH = ROOT_DIR / "models" / "modelo_seleccionado_cdcv_20260826.joblib"
joblib.dump(best_pipeline, MODEL_PATH)
print("Modelo guardado en:", MODEL_PATH)

## 13. Interpretación, análisis y recomendaciones

- El modelo seleccionado supera el baseline en la métrica principal (Recall).
- El pipeline integrado evita fuga de información porque encapsula imputación, escalado/encoding y modelo.
- La curva de aprendizaje permite verificar estabilidad del recall al aumentar datos y detectar posibles brechas train/validación.
- Si en futuras iteraciones se observa caída del recall en datos nuevos, se recomienda repetir feature engineering y re-entrenamiento con validación temporal/externa.
